<a href="https://colab.research.google.com/github/Trickwillfrit/EPFL_Rolex/blob/main/notebooks/evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluation of recommendation models

In this notebook, we build the evaluation pipeline for the recommendation system.

Our goals are:
- create a validation setup from the training interactions
- define the ground-truth items for each user
- implement Precision@10 and Recall@10
- compare recommendation models fairly

In [10]:
import pandas as pd
import matplotlib.pyplot as plt

BASE_URL = "https://raw.githubusercontent.com/Trickwillfrit/EPFL_Rolex/main/"

interactions = pd.read_csv(BASE_URL + "interactions_train.csv")
items = pd.read_csv(BASE_URL + "items.csv")
submission = pd.read_csv(BASE_URL + "sample_submission.csv")

print("Interactions shape:", interactions.shape)
display(interactions.head())

Interactions shape: (87047, 3)


,u,i,t
0,4456,8581,1.687541e+09
1,142,1964,1.679585e+09
2,362,3705,1.706872e+09
3,1809,11317,1.673533e+09
4,4384,1323,1.681402e+09


In [11]:
print("Interactions shape:", interactions.shape)
display(interactions.head())
print(interactions.columns)

Interactions shape: (87047, 3)


,u,i,t
0,4456,8581,1.687541e+09
1,142,1964,1.679585e+09
2,362,3705,1.706872e+09
3,1809,11317,1.673533e+09
4,4384,1323,1.681402e+09


Index(['u', 'i', 't'], dtype='object')


In [12]:
# Convert the timestamp column into a readable datetime format
interactions["datetime"] = pd.to_datetime(interactions["t"], unit="s")

# Sort interactions by user and time
interactions = interactions.sort_values(["u", "datetime"]).reset_index(drop=True)

# Quick check
display(interactions.head())

,u,i,t,datetime
0,0,0,1.680191e+09,2023-03-30 15:44:30
1,0,1,1.680783e+09,2023-04-06 12:13:54
2,0,2,1.680801e+09,2023-04-06 17:15:08
3,0,3,1.683715e+09,2023-05-10 10:35:45
4,0,3,1.683715e+09,2023-05-10 10:35:50


In [13]:
# Count the number of interactions for each user
user_counts = interactions["u"].value_counts()

# Display summary statistics
print(user_counts.describe())

count    7838.000000
mean       11.105767
std        16.441875
min         3.000000
25%         3.000000
50%         6.000000
75%        11.000000
max       385.000000
Name: count, dtype: float64


## Validation strategy

To evaluate our recommendation models, we use a leave-one-out strategy.

For each user, we keep the **last interaction in time** as the validation target, and we use all previous interactions as the training history.

This setup is appropriate for recommendation because it simulates a realistic situation: we try to predict a future item from the user's past behavior.

## What is the validation
The validation set is the part of the data that we keep aside to evaluate the model.

The model is trained only on the training set, and then we test whether it can correctly recommend the items contained in the validation set.

In [14]:
# For each user, keep the last interaction as validation
valid = interactions.groupby("u").tail(1).copy()

# Keep all previous interactions as training
train = interactions.drop(valid.index).copy()

# Reset indices for cleaner tables
train = train.reset_index(drop=True)
valid = valid.reset_index(drop=True)

# Check the sizes of the two splits
print("Train shape:", train.shape)
print("Validation shape:", valid.shape)

# Show the first rows
display(train.head())
display(valid.head())

Train shape: (79209, 4)
Validation shape: (7838, 4)


,u,i,t,datetime
0,0,0,1.680191e+09,2023-03-30 15:44:30
1,0,1,1.680783e+09,2023-04-06 12:13:54
2,0,2,1.680801e+09,2023-04-06 17:15:08
3,0,3,1.683715e+09,2023-05-10 10:35:45
4,0,3,1.683715e+09,2023-05-10 10:35:50


,u,i,t,datetime
0,0,24,1.713272e+09,2024-04-16 12:58:06
1,1,39,1.707759e+09,2024-02-12 17:37:24
2,2,92,1.709921e+09,2024-03-08 17:59:25
3,3,172,1.714129e+09,2024-04-26 11:01:00
4,4,195,1.714578e+09,2024-05-01 15:35:37


In [15]:
# Build the ground-truth dictionary from the validation set
# Each user is associated with the item we want the model to recover
ground_truth = valid.groupby("u")["i"].apply(list).to_dict()

# Build the training history dictionary
# For each user, store the set of items seen in training
train_user_items = train.groupby("u")["i"].apply(set).to_dict()

# Quick checks
print("Number of users in ground truth:", len(ground_truth))
print("Number of users in train history:", len(train_user_items))

# Show one example
example_user = list(ground_truth.keys())[0]
print("Example user:", example_user)
print("Train items:", train_user_items[example_user])
print("Validation item:", ground_truth[example_user])

Number of users in ground truth: 7838
Number of users in train history: 7838
Example user: 0
Train items: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23}
Validation item: [24]


In [16]:
# Function to compute Precision@K for one user
def precision_at_k(recommended_items, true_items, k=10): #among the top 10 books recommended by the model, how many are actually correct?
    # Keep only the first k recommended items
    recommended_items = recommended_items[:k]

    # Count how many recommended items are actually relevant
    hits = len(set(recommended_items) & set(true_items))

    # Precision@K = number of relevant recommended items / k
    return hits / k

In [17]:
# Function to compute Recall@K for one user
def recall_at_k(recommended_items, true_items, k=10):
    # Keep only the first k recommended items
    recommended_items = recommended_items[:k]

    # Count how many recommended items are actually relevant
    hits = len(set(recommended_items) & set(true_items))

    # Recall@K = number of relevant recommended items / number of true relevant items
    return hits / len(true_items)

In [18]:
# Function to evaluate a recommender over all users
def evaluate_recommender(recommendations, ground_truth, k=10):
    precision_scores = []
    recall_scores = []

    # Loop over all users in the ground truth
    for user, true_items in ground_truth.items():
        # Get the recommended items for this user
        recommended_items = recommendations.get(user, [])

        # Compute Precision@K and Recall@K for this user
        precision_scores.append(precision_at_k(recommended_items, true_items, k))
        recall_scores.append(recall_at_k(recommended_items, true_items, k))

    # Return the average scores across users
    return {
        "Precision@K": sum(precision_scores) / len(precision_scores),
        "Recall@K": sum(recall_scores) / len(recall_scores)
    }

In [19]:
# Create a small fake recommendation dictionary for a few users
fake_recommendations = {
    0: [5, 8, 24, 31, 44, 10, 3, 2, 90, 11],   # contains the true item 24
    1: [7, 12, 19, 25, 39, 41, 50, 60, 70, 80], # contains the true item 39
    2: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]          # does not contain the true item 92
}

# Create the matching fake ground truth
fake_ground_truth = {
    0: [24],
    1: [39],
    2: [92]
}

# Evaluate the fake recommender
fake_results = evaluate_recommender(fake_recommendations, fake_ground_truth, k=10)
print(fake_results)

{'Precision@K': 0.06666666666666667, 'Recall@K': 0.6666666666666666}


## Understanding Precision@10 and Recall@10

In this notebook, each user has **one validation item**, which is the last book they interacted with.

When we say that a model **recovers** a book, we simply mean that the true validation book appears somewhere in the top-10 recommendation list.

### Precision@10
Precision@10 answers the question:

**Among the 10 books recommended by the model, how many are correct?**

Since we always recommend 10 books, the denominator is 10.

For example, if the true validation book is in the recommendation list, then there is 1 correct item out of 10:

Precision@10 = 1 / 10 = 0.1

If the true book is not in the list, then:

Precision@10 = 0 / 10 = 0

### Recall@10
Recall@10 answers the question:

**Among the true relevant books, how many did the model recover?**

In our current setup, each user has only **one true validation book**, so the denominator is 1.

If the model recommends that true book, then:

Recall@10 = 1 / 1 = 1

If the model does not recommend it, then:

Recall@10 = 0 / 1 = 0

### Why the denominators are different
- Precision uses the number of recommended items, because it measures the quality of the recommendation list.
- Recall uses the number of true relevant items, because it measures whether the model managed to recover the relevant items.

So in our leave-one-out setup:
- if the true book is in the top 10, then Precision@10 = 0.1 and Recall@10 = 1
- otherwise, both metrics are 0

In [20]:
# Count item popularity in the training set
item_popularity = train["i"].value_counts()

# Keep the item IDs ordered by popularity
popular_items = item_popularity.index.tolist()

# Build recommendations for each user:
# recommend the most popular items that the user has not already seen
popular_recommendations = {}

for user in ground_truth.keys():
    seen_items = train_user_items[user]

    # Filter out items already seen by the user
    recs = [item for item in popular_items if item not in seen_items][:10]

    popular_recommendations[user] = recs

# Show recommendations for one example user
example_user = list(popular_recommendations.keys())[0]
print("Example user:", example_user)
print("Recommended items:", popular_recommendations[example_user])
print("True validation item:", ground_truth[example_user])

Example user: 0
Recommended items: [3055, 11366, 10715, 8999, 611, 4426, 53, 2820, 14555, 14578]
True validation item: [24]
